# 09 - Logit-based causal tracing, TF-IDF (GPU, ~1 h)

**Settings:** GPU T4 x1 | Internet On | Secret `HF_TOKEN` (optional).

The mid-term slides' causal tracing (v3 notebook: fraction of the clean-vs-CF log-prob gap of hop 1 recovered), on the same 219 TF-IDF pairs as notebooks 05/08, all 28 blocks: entity token (last), last prompt token, MLP window (±2), plus entity span and tail. Preflight = 2 pairs; check `sanity` ~1 and that recovery values are between ~0 and ~1. If CUDA OOM: lower `--batch` (finished pairs are kept).

Download `causal_tracing_tfidf.zip` (summary.md, causal_trace.png, pairs.jsonl).

In [ ]:
# ===== Setup: internet, clone, HF token, GPU =====
import os, socket, subprocess, sys, time, zipfile, json
for host in ('github.com', 'huggingface.co'):
    socket.gethostbyname(host)
WORK = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'cf-tfidf', 'https://github.com/ANLPproject/LinguaFranca.git', WORK], check=True)
else:
    subprocess.run(['git', '-C', WORK, 'pull'], check=True)
os.chdir(WORK); sys.path.insert(0, WORK)
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')    # read by the scripts from the environment
    print('HF_TOKEN loaded from Kaggle secrets')
except Exception as e:
    print(f'No HF_TOKEN secret ({e.__class__.__name__}); the ungated model mirror will be used')
import torch
assert torch.cuda.is_available(), 'No GPU: Session options -> Accelerator -> GPU T4 x1, then restart.'

def run(cmd):
    '''Stream a command's output; return its exit code.'''
    print('$', ' '.join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        if 'it/s]' in line or 's/it]' in line:
            continue
        print(line, end='')
    return p.wait()

OUT = '/kaggle/working/causal_tracing_tfidf'
PRE = '/kaggle/working/causal_tracing_preflight'

In [ ]:
# ===== PREFLIGHT (2 pairs) =====
rc = run([sys.executable, 'causal_tracing_logit.py', '--out', PRE, '--max-pairs', '2'])
assert rc == 0 and os.path.exists(f'{PRE}/summary.md'), 'PREFLIGHT FAILED - read the log above.'
print('PREFLIGHT PASSED')

In [ ]:
# ===== MAIN RUN: the 219 pairs of notebooks 05/08 (resumable) =====
rc = run([sys.executable, 'causal_tracing_logit.py', '--out', OUT, '--max-pairs', '219', '--max-minutes', '600'])
print('exit code:', rc)

In [ ]:
# ===== package (always run) =====
run([sys.executable, 'causal_tracing_logit.py', '--out', OUT, '--summarize-only'])
zpath = '/kaggle/working/causal_tracing_tfidf.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in (OUT, PRE):
        for root, _, files in os.walk(d):
            for f in files:
                p = os.path.join(root, f)
                z.write(p, os.path.relpath(p, os.path.dirname(d)))
print(f'{zpath}: {os.path.getsize(zpath) / 1e6:.1f} MB')